In [58]:
import pandas as pd
import re # regex
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [28]:
df = pd.read_csv("X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\RNN\\IMDB Dataset.csv")

In [29]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [30]:
df.isna().sum()

review       0
sentiment    0
dtype: int64

In [31]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

## Data PreProcessing 

1. Converting to Lowercase

In [32]:
df["review"] = df["review"].str.lower()

2. Removing URLs 

In [33]:
def remove_url(text):
    text = re.sub(r"http\S+", "", text) # Pattern, Replacement, string
    # \S -> Non-white spaces
    # \s -> white spaces
    return text
df["review"] = df["review"].apply(remove_url)

3. Removing punctuations

In [34]:
def remoce_punc(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remoce_punc)

4. Remove Tags

In [35]:
def remoce_tags(text):
    text = re.sub(r"<.*?>", "", text) 
    return text

df["review"] = df["review"].apply(remoce_tags)

5. Removing Stop Works

In [36]:
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

In [37]:
# NLTK Example

# sample_text = "I love Coding in Python"
# token = word_tokenize(sample_text)
# token

# output - ['I', 'love', 'Coding', 'in', 'Python']


In [38]:
def remove_stopword(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))

    return " ".join(word for word in tokens if word not in stop_words)

df["review"] = df["review"].apply(remove_stopword)

In [39]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


6. Stemming 

In [40]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words) 

df["review"] = df["review"].apply(stemming)

7. Encoding

In [41]:
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]

In [42]:
y 
# converted Positive value to 1
# converted Negative value to 0

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

8. Vectorization 

In [46]:
tf = TfidfVectorizer(
    max_features=5000
)

X = tf.fit_transform(df["review"])

## Dataset and Data Loaders

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=42
)


In [48]:
X_train.shape

(39665, 5000)

In [49]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [51]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

C:\Users\Rajat Bhardwaj\AppData\Local\Temp\ipykernel_29684\2565067486.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()


In [53]:
train_loader = DataLoader(
    train_set, 
    shuffle=True,
    batch_size=64
)
test_loader = DataLoader(
    test_set, 
    shuffle=True,
    batch_size=64
)

## RNN - Recurrent Neural Network

In [62]:
# Many to One architecture 
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
       # RNN Layer
        self.rnn = nn.RNN(
            input_size,
            hidden_size, 
            num_layers, 
            batch_first=True
        )

        # Fully Connected Layer

        self.fc = nn.Linear(hidden_size, 1)


    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timestep 
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [63]:
input_size  = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training the model

In [65]:
epochs = 10

for epoch in range (epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        output = model(Xb)
        output = torch.sigmoid(output.squeeze()) # probability

        loss = criterion(output, yb) # compute loss
        loss.backward()
        optimizer.step()

    print(f"Epoch: {epoch+1}/{epochs} and Loss: {loss.item()}")

Epoch: 1/10 and Loss: 0.3443469703197479
Epoch: 2/10 and Loss: 0.35370340943336487
Epoch: 3/10 and Loss: 0.19734159111976624
Epoch: 4/10 and Loss: 0.14933091402053833
Epoch: 5/10 and Loss: 0.26614221930503845
Epoch: 6/10 and Loss: 0.17284268140792847
Epoch: 7/10 and Loss: 0.07016933709383011
Epoch: 8/10 and Loss: 0.3522295355796814
Epoch: 9/10 and Loss: 0.17211633920669556
Epoch: 10/10 and Loss: 0.2872539758682251


In [70]:
model.eval()

with torch.no_grad():
    correct_values = 0
    total_values = 0
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        output = model(Xb)
        predicted = (torch.sigmoid(output.squeeze()) > 0.5).float()

        total_values += yb.size(0)
        correct_values += (predicted == yb).sum().item()

    print(f"Accuracy: {correct_values/total_values*100}")

Accuracy: 87.40546536250882
